In [72]:
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
import torch.optim as optim
import pickle
import numpy as np

In [ ]:
def custom_collate_fn(batch):
    # Find the maximum number of 'CSFs' across all samples in the batch.
    max_csf_count = max(item['CSFs'].size(0) for item in batch)
    
    # Initialize lists to hold padded 'CSFs' and the original lengths.
    padded_csfs_list = []
    original_lengths = []
    
    for item in batch:
        original_length = item['CSFs'].size(0)
        original_lengths.append(original_length)
        padded_csfs = torch.cat([
            item['CSFs'], 
            torch.zeros(max_csf_count - original_length, 4)  # Pad with zeros.
        ])
        padded_csfs_list.append(padded_csfs)
        
    # Stack the padded 'CSFs'.
    padded_csfs = torch.stack(padded_csfs_list)
    
    # Create a mask based on the original lengths.
    mask = torch.ones_like(padded_csfs[:, :, 0], dtype=torch.bool)
    for i, length in enumerate(original_lengths):
        mask[i, :length] = False
    
    # Convert other attributes to tensors.
    n_electrons = torch.tensor([item['n_electrons'] for item in batch])
    n_protons = torch.tensor([item['n_protons'] for item in batch])
    runtime = torch.tensor([item['runtime'] for item in batch])
    convergence = torch.tensor([item['convergence'] for item in batch])
    effect = torch.tensor([item['effect'] for item in batch])
    
    # Return a dictionary with the batched data and the mask.
    return {
        'CSFs': padded_csfs,
        'mask': mask,
        'n_electrons': n_electrons,
        'n_protons': n_protons,
        'runtime': runtime,
        'convergence': convergence,
        'effect': effect
    }

In [2]:
class CustomDataset(Dataset):
    def __init__(self, data):
        """
        data: The nested dictionary containing all your data
        """
        self.data = data
        self.index_mapping = self._create_index_mapping()
    
    def _create_index_mapping(self):
        mapping = []
        for n_key, csfs_dict in self.data.items():
            for csf_key, details in csfs_dict.items():
                mapping.append((n_key, csf_key))
        return mapping

    def __len__(self):
        return len(self.index_mapping)

    def __getitem__(self, idx):
        # Retrieve the actual keys from the index
        n_key, csf_key = self.index_mapping[idx]
        
        # Access the data
        data_point = self.data[n_key][csf_key]

class GraspDataset(Dataset):
    def __init__(self, grasp_manager):
        self.data = []
        excitations = set().union(*[x.excitations for x in grasp_manager.cache.keys()])

        for CSF, result in grasp_manager.cache.items():

            if result.error:
                continue

            tree = scipy.spatial.KDTree(np.vstack([result.eigenvectors, -result.eigenvectors]))

            for excitation in excitations - CSF.excitations:
                new_CSF = CSF.add_excitation(excitation)

                if new_CSF not in grasp_manager.cache:
                    continue
                new_index = sorted(new_CSF.excitations).index(excitation)
                runtime, convergence, effect = process_result(new_index, tree, grasp_manager.cache[new_CSF])

                self.data.append({
                    "CSFs": torch.tensor([excitation] + list(CSF.excitations)),
                    "n_electrons": CSF.num_electrons,
                    "n_protons": CSF.num_protons,
                    "runtime": runtime,
                    "convergence": convergence,
                    "effect": effect
                })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

In [2]:
data_dict_path = '/home/projects/ku_00258/people/mouhol/METAL-AI/data/Metal_data_dict_dataset/second_test_data_dict.pkl'

# Use 'rb' to read in binary mode
with open(data_dict_path, 'rb') as file:
    data_dict = pickle.load(file)

In [87]:
class CustomDataset(Dataset):
    def __init__(self, data, remove_nan_effect=False):
        """
        data: The nested dictionary containing all your data
        """
        self.data = data
        self.index_mapping = self._create_index_mapping(remove_nan_effect)
    
    def _create_index_mapping(self, remove_nan_effect):
        mapping = []
        for ion_key in self.data.keys():
            for asf_key in self.data[ion_key].keys():
                if remove_nan_effect:
                    if 'effect' not in self.data[ion_key][asf_key]:
                        continue
                    if np.isnan(self.data[ion_key][asf_key]['effect']).all():
                        continue
                mapping.append((ion_key, asf_key))
        return mapping

    def __len__(self):
        return len(self.index_mapping)

    def __getitem__(self, idx):
        # Retrieve the actual keys from the index
        ion_key, asf_key = self.index_mapping[idx]
        
        # Access the data
        data_point = self.data[ion_key][asf_key]

        data_point['excitations'] = torch.tensor(data_point['excitations'])
        if 'effect' not in data_point:
            data_point['effect'] = torch.full((data_point['excitations'].size(0),), float('nan'))
        else:
            data_point['effect'] = torch.tensor(data_point['effect'])


        data_point['Converged'] = torch.tensor(data_point['Converged'], dtype=torch.bool)
        # data_point['excitations'] = torch.tensor(data_point['excitations'])
        data_point['index'] = torch.tensor(data_point['index'], dtype=torch.long)
        
        data_point['n_protons'] = torch.tensor(ion_key[0], dtype=torch.long)
        data_point['n_electrons'] = torch.tensor(ion_key[1], dtype=torch.long)
        
        # data_point['n_protons'] = ion_key[0]
        # data_point['n_electrons'] = ion_key[1]
        
        return data_point

In [4]:
for ion_key in data_dict.keys():
    for asf_key in data_dict[ion_key].keys():
        if 'excitations' not in data_dict[ion_key][asf_key]:

            print(ion_key, asf_key,data_dict[ion_key][asf_key])

In [73]:
for ion_key in data_dict.keys():
    for asf_key in data_dict[ion_key].keys():
        if 'effect' in data_dict[ion_key][asf_key]:
            if np.isnan(data_dict[ion_key][asf_key]['effect']).all():
                print(ion_key, asf_key,data_dict[ion_key][asf_key])

            

(6, 4) ((1, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0), (2, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0), (2, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0), (2, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0), (2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)) {'excitations': tensor([[0, 1, 1, 1],
        [0, 0, 2, 2],
        [0, 0, 0, 4],
        [0, 0, 0, 3],
        [0, 0, 0, 0]]), 'index': 1, 'Converged': False, 'effect': tensor([nan, nan, nan, nan, nan], dtype=torch.float64), 'n_protons': 6, 'n_electrons': 4}
(6, 4) ((1, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0), (1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0), (2, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0), (2, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0), (2, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0), (2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)) {'excitations': tensor([[0, 1, 1, 1],
    

In [79]:
dataset = CustomDataset(data_dict, remove_nan_effect=True)
print(dataset[0])

{'excitations': tensor([[1, 1, 1, 1],
        [1, 0, 0, 7],
        [1, 0, 0, 1],
        [0, 0, 1, 1],
        [0, 0, 0, 3],
        [0, 0, 0, 0]]), 'index': tensor(1), 'Converged': True, 'effect': tensor([1.0016, 0.0067, 0.0021, 0.0072, 0.9972,    nan], dtype=torch.float64), 'n_protons': 6, 'n_electrons': 4}


<ipython-input-78-ab15911b5b23>:34: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  data_point['excitations'] = torch.tensor(data_point['excitations'])
<ipython-input-78-ab15911b5b23>:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  data_point['effect'] = torch.tensor(data_point['effect'])


In [94]:
dataset = CustomDataset(data_dict, remove_nan_effect=False)
for data in dataset:
    if torch.isnan(data['effect']).all():
        print(data)

<ipython-input-87-3f09ff6ce00d>:34: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  data_point['excitations'] = torch.tensor(data_point['excitations'])
<ipython-input-87-3f09ff6ce00d>:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  data_point['effect'] = torch.tensor(data_point['effect'])
<ipython-input-87-3f09ff6ce00d>:47: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  data_point['index'] = torch.tensor(data_point['index'], dtype=torch.long)


{'excitations': tensor([[0, 1, 1, 1],
        [0, 0, 2, 2],
        [0, 0, 0, 4],
        [0, 0, 0, 3],
        [0, 0, 0, 0]]), 'index': tensor(1), 'Converged': tensor(False), 'effect': tensor([nan, nan, nan, nan, nan], dtype=torch.float64), 'n_protons': tensor(6), 'n_electrons': tensor(4)}
{'excitations': tensor([[0, 1, 1, 1],
        [1, 0, 0, 7],
        [0, 0, 2, 2],
        [0, 0, 0, 4],
        [0, 0, 0, 3],
        [0, 0, 0, 0]]), 'index': tensor(2), 'Converged': tensor(False), 'effect': tensor([nan, nan, nan, nan, nan, nan]), 'n_protons': tensor(6), 'n_electrons': tensor(4)}
{'excitations': tensor([[0, 1, 1, 1],
        [1, 0, 0, 1],
        [0, 0, 1, 1],
        [0, 0, 0, 4],
        [0, 0, 0, 2],
        [0, 0, 0, 0]]), 'index': tensor(4), 'Converged': tensor(False), 'effect': tensor([nan, nan, nan, nan, nan, nan]), 'n_protons': tensor(6), 'n_electrons': tensor(4)}
{'excitations': tensor([[1, 0, 0, 7],
        [0, 0, 0, 0]]), 'index': tensor(0), 'Converged': tensor(False), 'e

In [80]:
for data in dataset:
    if 'effect' not in data:
        print(data['Converged'])

<ipython-input-78-ab15911b5b23>:34: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  data_point['excitations'] = torch.tensor(data_point['excitations'])
<ipython-input-78-ab15911b5b23>:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  data_point['effect'] = torch.tensor(data_point['effect'])
<ipython-input-78-ab15911b5b23>:47: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  data_point['index'] = torch.tensor(data_point['index'], dtype=torch.long)


In [95]:
def custom_collate_fn(batch):
    # Find the maximum number of 'CSFs' across all samples in the batch.
    # print(len(item) for item in batch)
    max_csf_count = max([len(item['excitations']) for item in batch])
    
    # Initialize lists to hold padded 'CSFs' and the original lengths.
    padded_csfs_list = []
    original_lengths = []
    padded_effect_list = []
    
    for item in batch:
        original_length = item['excitations'].size(0)
        original_lengths.append(original_length)
        padded_csfs = torch.cat([
            item['excitations'], 
            torch.zeros(max_csf_count - original_length, 4)  # Pad with zeros.
        ])
        padded_effect = torch.cat([
            item['effect'], 
            torch.zeros(max_csf_count - original_length)  # Pad with zeros.
        ])
        padded_csfs_list.append(padded_csfs)
        padded_effect_list.append(padded_effect)
        
    # Stack the padded 'CSFs'.
    padded_csfs = torch.stack(padded_csfs_list)
    padded_effect = torch.stack(padded_effect_list)
    
    # Create a mask based on the original lengths.
    mask = torch.ones_like(padded_csfs[:, :, 0], dtype=torch.bool)
    for i, length in enumerate(original_lengths):
        mask[i, :length] = False
    
    # Convert other attributes to tensors.
    n_electrons = torch.stack([item['n_electrons'] for item in batch])
    n_protons = torch.stack([item['n_protons'] for item in batch])
    convergence = torch.stack([item['Converged'] for item in batch])
    # n_electrons = torch.tensor([item['n_electrons'] for item in batch])
    # n_protons = torch.tensor([item['n_protons'] for item in batch])
    # convergence = torch.tensor([item['Converged'] for item in batch])
    # effect = torch.tensor([item['effect'] for item in batch])
    
    # Return a dictionary with the batched data and the mask.
    return {
        'excitations': padded_csfs,
        'mask': mask,
        'n_electrons': n_electrons,
        'n_protons': n_protons,
        'convergence': convergence,
        'effect': padded_effect
    }

In [96]:
dataloader = DataLoader(dataset, batch_size=128, collate_fn=custom_collate_fn)

# Iterate through the DataLoader and print the first batch.
for batch in dataloader:
    print(batch['excitations'].shape)
    print(batch['effect'].shape)
    print(batch['mask'].shape)
    # print(batch)
    # break

torch.Size([128, 7, 4])
torch.Size([128, 7])
torch.Size([128, 7])
torch.Size([128, 7, 4])
torch.Size([128, 7])
torch.Size([128, 7])


<ipython-input-87-3f09ff6ce00d>:34: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  data_point['excitations'] = torch.tensor(data_point['excitations'])
<ipython-input-87-3f09ff6ce00d>:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  data_point['effect'] = torch.tensor(data_point['effect'])
<ipython-input-87-3f09ff6ce00d>:45: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  data_point['Converged'] = torch.tensor(data_point['Converged'], dtype=torch.bool)
<ipython-input-87-3f09ff6ce00d>:47: UserWarning: To copy construct from a tensor, it is re

torch.Size([128, 7, 4])
torch.Size([128, 7])
torch.Size([128, 7])
torch.Size([128, 7, 4])
torch.Size([128, 7])
torch.Size([128, 7])
torch.Size([128, 7, 4])
torch.Size([128, 7])
torch.Size([128, 7])
torch.Size([128, 7, 4])
torch.Size([128, 7])
torch.Size([128, 7])
torch.Size([128, 7, 4])
torch.Size([128, 7])
torch.Size([128, 7])
torch.Size([128, 7, 4])
torch.Size([128, 7])
torch.Size([128, 7])
torch.Size([128, 7, 4])
torch.Size([128, 7])
torch.Size([128, 7])
torch.Size([128, 7, 4])
torch.Size([128, 7])
torch.Size([128, 7])
torch.Size([128, 6, 4])
torch.Size([128, 6])
torch.Size([128, 6])
torch.Size([128, 6, 4])
torch.Size([128, 6])
torch.Size([128, 6])
torch.Size([128, 5, 4])
torch.Size([128, 5])
torch.Size([128, 5])
torch.Size([128, 7, 4])
torch.Size([128, 7])
torch.Size([128, 7])
torch.Size([128, 7, 4])
torch.Size([128, 7])
torch.Size([128, 7])
torch.Size([128, 7, 4])
torch.Size([128, 7])
torch.Size([128, 7])
torch.Size([128, 7, 4])
torch.Size([128, 7])
torch.Size([128, 7])
torch.Size

In [50]:
class CSF_encoder(nn.Module):
    def __init__(self, output_size=32):
        super(CSF_encoder, self).__init__()
        # Assuming the input size is 6 because we append n_electrons (1) and n_protons (1) to each 4-dimensional CSF.
        self.network = nn.Sequential(
        nn.Linear(6, 64),
        nn.ReLU(),
        nn.Linear(64, output_size)
        )

    def forward(self, csfs, n_electrons, n_protons):
        # Append n_electrons and n_protons to each CSF
        n_electrons = n_electrons.unsqueeze(-1).unsqueeze(-1).expand(-1, csfs.size(1), 1)
        n_protons = n_protons.unsqueeze(-1).unsqueeze(-1).expand(-1, csfs.size(1), 1)
        extended_csfs = torch.cat([csfs, n_electrons, n_protons], dim=-1)
        return self.network(extended_csfs)

class simple_GRASP_transformer_surrogate_model(nn.Module):
    def __init__(
        self, 
        csf_encoder,         
        input_size: int = 6,
        d_model: int = 64, 
        nhead: int = 8,
        dim_forward: int = 64,
        num_layers: int = 6,
        output_size = 1, 
        dropout=0.5
    ):
        super(simple_GRASP_transformer_surrogate_model, self).__init__()
        self.csf_encoder = csf_encoder
        encoder_layers = nn.TransformerEncoderLayer(d_model, nhead, dim_forward, dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        self.decoder = nn.Linear(d_model, output_size)


    def forward(self, input_dict):
        csfs = input_dict['excitations']
        encoded_csfs = self.csf_encoder(csfs, input_dict['n_electrons'], input_dict['n_protons'])
        output = self.transformer_encoder(encoded_csfs,src_key_padding_mask = input_dict['mask'])#src_key_padding_mask = ~input_dict['mask']
        # output = output[:, 0, :]
        output = self.decoder(output)
        return F.softplus(output).squeeze(-1)

# Example of creating and using these modules

d_model= 8
csf_encoder = CSF_encoder(d_model)
model = simple_GRASP_transformer_surrogate_model(csf_encoder,d_model=d_model)

In [51]:
for batch in dataloader:
    output= model(batch)
    print()
    print(output.shape)

<ipython-input-22-630c777b25be>:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  data_point['excitations'] = torch.tensor(data_point['excitations'])
<ipython-input-22-630c777b25be>:35: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  data_point['effect'] = torch.tensor(data_point['effect'])



torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 6])

torch.Size([128, 6])

torch.Size([128, 5])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 7])

torch.Size([128, 6])

torch.Size([128, 6])

torch.Size([128, 6])

torch.Size([128, 6])

torch.Size([128, 6])

torch.Siz

In [69]:
import torch
import torch.nn.functional as F



class Custom_loss(nn.Module):
    def __init__(self, loss_function=F.mse_loss):
        super(Custom_loss, self).__init__()
        self.loss_function = loss_function
        
    def forward(self, pred, target, mask):
        mask = mask.bool()
        valid_pred = ~torch.isnan(pred)
        valid_target = ~torch.isnan(target)
        valid_elements = valid_pred & valid_target & ~mask
        filtered_pred = pred[valid_elements]
        filtered_target = target[valid_elements]
        loss = F.mse_loss(filtered_pred, filtered_target)
        if loss.numel() > 1:
            loss = loss.sum()
            # print(loss)
        return loss

# (pred, target, mask, loss_function=F.mse_loss):
#     """
#     Calculates a custom loss by applying a given loss function to the elements of pred and target tensors
#     that are not NaN and not masked.

#     Parameters:
#     - pred: Predicted tensor.
#     - target: Target tensor.
#     - mask: A mask tensor where True indicates elements to be ignored.
#     - loss_function: A PyTorch loss function to apply to the unmasked, non-NaN elements.

#     Returns:
#     - A scalar tensor representing the loss.
#     """

#     # Ensure mask is a boolean tensor
#     mask = mask.bool()

#     # Find non-NaN elements in both pred and target
#     valid_pred = ~torch.isnan(pred)
#     valid_target = ~torch.isnan(target)
#     # Combine conditions: elements should be non-NaN in both tensors and not masked
#     valid_elements = valid_pred & valid_target & ~mask
#     # Apply the mask and select valid elements
#     filtered_pred = pred[valid_elements]
#     filtered_target = target[valid_elements]

#     # Compute the loss on the filtered elements using the provided loss function
#     loss = loss_function(filtered_pred, filtered_target)

#     # Sum the loss if it's not a single value
#     if loss.numel() > 1:
#         loss = loss.sum()

#     return loss


In [ ]:
class Custom_loss_converged(nn.Module):
    def __init__(self, loss_function=F.cross_entropy):
        super(Custom_loss, self).__init__()
        self.loss_function = loss_function
        
    def forward(self, pred, target, converge_index):
        loss = self.loss_function(pred[converge_index], target[converge_index])
        return loss

In [70]:
loss_func = Custom_loss()
for batch in dataloader:
    output= model(batch)
    print(output.dtype)
    loss = loss_func(output, batch['effect'], batch['mask'])
    # print(batch['effect'], batch['mask'],loss)
    print(loss)

<ipython-input-22-630c777b25be>:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  data_point['excitations'] = torch.tensor(data_point['excitations'])
<ipython-input-22-630c777b25be>:35: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  data_point['effect'] = torch.tensor(data_point['effect'])


torch.float32
tensor(0.4113, dtype=torch.float64, grad_fn=<MseLossBackward0>)
torch.float32
tensor(0.4304, dtype=torch.float64, grad_fn=<MseLossBackward0>)
torch.float32
tensor(0.4389, dtype=torch.float64, grad_fn=<MseLossBackward0>)
torch.float32
tensor(0.4587, dtype=torch.float64, grad_fn=<MseLossBackward0>)
torch.float32
tensor(0.4470, dtype=torch.float64, grad_fn=<MseLossBackward0>)
torch.float32
tensor(0.4280, dtype=torch.float64, grad_fn=<MseLossBackward0>)
torch.float32
tensor(0.4862, dtype=torch.float64, grad_fn=<MseLossBackward0>)
torch.float32
tensor(0.4103, dtype=torch.float64, grad_fn=<MseLossBackward0>)
torch.float32
tensor(0.4318, dtype=torch.float64, grad_fn=<MseLossBackward0>)
torch.float32
tensor(0.4129, dtype=torch.float64, grad_fn=<MseLossBackward0>)
torch.float32
tensor(0.4648, dtype=torch.float64, grad_fn=<MseLossBackward0>)
torch.float32
tensor(0.4315, dtype=torch.float64, grad_fn=<MseLossBackward0>)
torch.float32
tensor(0.4254, dtype=torch.float64, grad_fn=<MseLo

In [97]:
loss_func = custom_loss()  # Initialize the custom loss function
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Initialize the optimizer

num_epochs = 10  # Number of epochs to train for

for epoch in range(num_epochs):
    model.train()  # Set the model to training mode
    running_loss = 0.0

    for batch in dataloader:
        # Assuming your DataLoader outputs a batch as a dictionary with 'CSFs', 'n_electrons', 'n_protons', and 'effect'
        targets = batch['effect']  # The 'effect' tensor
        optimizer.zero_grad()  # Zero the parameter gradients

        # Forward pass
        outputs = model(batch)

        # Compute loss
        loss = loss_func(outputs.float(), batch['effect'].float(), batch['mask']) # Ensure targets is the correct shape
        # Backward pass and optimize
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch['effect'].size(0)  # Multiply by batch size for total loss

    epoch_loss = running_loss / len(dataloader.dataset)  # Average loss per sample
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")

print('Finished Training')

<ipython-input-87-3f09ff6ce00d>:34: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  data_point['excitations'] = torch.tensor(data_point['excitations'])
<ipython-input-87-3f09ff6ce00d>:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  data_point['effect'] = torch.tensor(data_point['effect'])
<ipython-input-87-3f09ff6ce00d>:45: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  data_point['Converged'] = torch.tensor(data_point['Converged'], dtype=torch.bool)
<ipython-input-87-3f09ff6ce00d>:47: UserWarning: To copy construct from a tensor, it is re

Epoch 1/10, Loss: 0.6410
Epoch 2/10, Loss: 0.6362
Epoch 3/10, Loss: 0.6415
Epoch 4/10, Loss: 0.6267
Epoch 5/10, Loss: 0.6252
Epoch 6/10, Loss: 0.6233
Epoch 7/10, Loss: 0.6171
Epoch 8/10, Loss: 0.6177
Epoch 9/10, Loss: 0.6214
Epoch 10/10, Loss: 0.6097
Finished Training
